In [50]:
import pandas as pd
import requests
import time
import os
import zipfile
import urllib.request
from tqdm import tqdm 
from dotenv import load_dotenv, find_dotenv


In [51]:

# Tự động tìm file .env ở các thư mục cha
load_dotenv(find_dotenv())

# Sử dụng biến bình thường
TMDB_API_KEY = os.getenv("TMDB_API_KEY")
print(TMDB_API_KEY)

de6ad9963ce22f788c1d2180429def70


In [ ]:
DATA_PATH = "../data"

In [ ]:
!pip install tqdm

In [ ]:
# !pip3 install kagglehub 

In [ ]:
# !pip install kagglehub[hf-datasets]

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("grouplens/movielens-latest-full")

print("Path to dataset files:", path)


In [ ]:
full_df = {
    "ratings": pd.read_csv(path + "/ratings.csv"),
    "tags": pd.read_csv(path + "/tags.csv"),
    "links": pd.read_csv(path + "/links.csv"),
    "movies": pd.read_csv(path + "/movies.csv"),
    "genome_scores": pd.read_csv(path + "/genome-scores.csv"),
    "genome_tags": pd.read_csv(path + "/genome-tags.csv")
}

In [ ]:
MOVIELENS_DIR = os.path.join(DATA_PATH, 'movielens')
if not os.path.exists(MOVIELENS_DIR):
    os.makedirs(MOVIELENS_DIR)
for name, df in full_df.items():
    print(f"{50*'='}")
    print(f"DataFrame '{name}':")
    print(df.head())
    
    save_path = os.path.join(MOVIELENS_DIR, f"{name}.csv")
    df.to_csv(save_path, index=False)
    print("\n")

In [ ]:
# 1. Nhóm theo movieId để tính Số lượt đánh giá và Điểm trung bình
movie_stats = full_df['ratings'].groupby('movieId').agg(
    rating_count=('rating', 'count'),
    rating_mean=('rating', 'mean')
).reset_index()

# 2. Áp dụng tiêu chuẩn lọc (Filter)
MIN_RATINGS = 50  # Phim phải có ít nhất 50 người rate
MIN_SCORE = 3.0   # Điểm trung bình từ 3.0 trở lên

quality_movies = movie_stats[
    (movie_stats['rating_count'] >= MIN_RATINGS) & 
    (movie_stats['rating_mean'] >= MIN_SCORE)
]

print(f"Số phim ban đầu: {len(movie_stats)}")
print(f"Số phim 'Đáng xem' sau khi lọc: {len(quality_movies)}")

# 3. Gộp (Merge) kết quả đã lọc với bảng links_df để lấy tmdbId
filtered_links_df = pd.merge(quality_movies, full_df['links'], on='movieId', how='inner')

display(filtered_links_df.sort_values(by='rating_count', ascending=False).head())

In [ ]:


def fetch_movie_vietnamese_metadata(tmdb_id):
    """Hàm gọi API TMDB để lấy thông tin phim bằng tiếng Việt"""
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&language=vi-VN"
    
    try:
        response = requests.get(url, timeout=10)
        # Nếu request thành công
        if response.status_code == 200:
            data = response.json()
            # print(f"Đã fetch thành công tmdbId {tmdb_id}: {data.get('title', '')}")
            return {
                'tmdbId': tmdb_id,
                'title_vi': data.get('title', ''),
                'overview_vi': data.get('overview', ''),
                # Lấy danh sách tên thể loại và nối thành chuỗi
                'genres': ", ".join([g['name'] for g in data.get('genres', [])]) if data.get('genres') else "",
                'release_date': data.get('release_date', ''),
                'poster_path': data.get('poster_path', '') # Rất hữu ích cho frontend sau này,
                
            } | data
        # Nếu phim không tồn tại trên TMDB
        elif response.status_code == 404:
            return None
    except Exception as e:
        print(f"Lỗi khi fetch tmdbId {tmdb_id}: {e}")
        return None
    return None

In [ ]:
tmdb_id = 550  # Ví dụ: tmdbId của phim "Fight Club"
metadata = fetch_movie_vietnamese_metadata(tmdb_id)
for k, v in metadata.items():
    print(f"{k}: {v}")

In [ ]:
movie_vi_data = []
    
# fetch theo đoạn: limit, offset
last_offset = 0
batch_size = 1000


def fetch_movie_batch(limit=1000, offset=0):
    """Hàm fetch một batch phim từ filtered_links_df"""
    batch_links = filtered_links_df.iloc[offset:offset+limit]
    batch_data = [] 
    
    for index, row in tqdm(batch_links.iterrows(), total=batch_links.shape[0], desc=f"Đang fetch batch {offset + 1}-{offset + limit}"):
        movie_id = row['movieId']
        tmdb_id = row['tmdbId']
        
        metadata = fetch_movie_vietnamese_metadata(tmdb_id)
        
        if metadata:
            metadata['movieId'] = movie_id
            batch_data.append(metadata)
        
        time.sleep(0.05)  # Tránh bị block API
    
    movie_vi_data.extend(batch_data)
    save_movie(batch_data, offset, limit)
    return batch_data, offset + limit

def save_movie(data, offset, limit, prefix="movies_vietnamese_metadata"):
    # Chuyển list các dictionary thành Pandas DataFrame
    movies_vi_df = pd.DataFrame(data)

    # Hiển thị vài dòng đầu để kiểm tra
    # display(movies_vi_df.head())

    # Lưu ra file CSV
    TMDB_DIR = os.path.join(DATA_PATH, 'tmdb')
    if not os.path.exists(TMDB_DIR):
        os.makedirs(TMDB_DIR)
        
    start_index = offset + 1
    end_index = offset + len(data)
    output_filename = os.path.join(TMDB_DIR, f"{prefix}_{start_index}-{end_index}.jsonl")
    movies_vi_df.to_json(output_filename, orient='records', lines=True, force_ascii=False)
    print(f"\nĐã lưu thành công {len(movies_vi_df)} phim tiếng Việt vào file {output_filename}!")




data, last_offset = fetch_movie_batch(limit=5, offset=last_offset)
data, last_offset = fetch_movie_batch(limit=5, offset=last_offset)

Đang fetch batch 1-5 (tổng 5 phim)


Đang fetch batch 1-5:  20%|██        | 1/5 [00:00<00:03,  1.11it/s]

Đã fetch thành công tmdbId 862.0: Câu Chuyện Đồ Chơi


Đang fetch batch 1-5:  40%|████      | 2/5 [00:01<00:02,  1.11it/s]

Đã fetch thành công tmdbId 8844.0: Jumanji: Trò Chơi Kỳ Ảo


...

Đã fetch thành công tmdbId 9091.0: Cái Chết Bất Ngờ


Đang fetch batch 6-10:  80%|████████  | 4/5 [00:03<00:00,  1.10it/s]

Đã fetch thành công tmdbId 710.0: Điệp Viên 007: Điệp Vụ Mắt Vàng


Đang fetch batch 6-10: 100%|██████████| 5/5 [00:04<00:00,  1.11it/s]

Đã fetch thành công tmdbId 9087.0: Tổng Thống Hoa Kỳ

Đã lưu thành công 5 phim tiếng Việt vào file ../data\tmdb\movies_vietnamese_metadata_6-10.jsonl!


In [62]:
save_movie(movie_vi_data, offset=0, limit=len(movie_vi_data))


Đã lưu thành công 10 phim tiếng Việt vào file ../data\tmdb\movies_vietnamese_metadata_1-10.jsonl!
